### Search Engine With Tools And Agents

In [34]:
# ============================================================
# Search Engine With Tools And Agents
# LangChain 1.x + LangGraph
# ============================================================

# ------------------------------------------------------------
# 1. Arxiv + Wikipedia Tools
# ------------------------------------------------------------

from langchain_community.tools import (
    ArxivQueryRun,
    WikipediaQueryRun
)
from langchain_community.utilities import (
    WikipediaAPIWrapper,
    ArxivAPIWrapper
)

In [35]:
# Wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

wiki = WikipediaQueryRun(
    api_wrapper=api_wrapper_wiki
)

print(wiki.name)

wikipedia


In [36]:
# Arxiv
api_wrapper_arxiv = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

arxiv = ArxivQueryRun(
    api_wrapper=api_wrapper_arxiv
)

print(arxiv.name)


tools = [wiki, arxiv]


arxiv


In [37]:
# ------------------------------------------------------------
# 2. Custom RAG Tool
# ------------------------------------------------------------

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.tools import create_retriever_tool

In [38]:
# ------------------------------------------------------------
# 2. Custom RAG Tool
# ------------------------------------------------------------

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.tools import create_retriever_tool


# Load LangSmith documentation
loader = WebBaseLoader(
    "https://docs.smith.langchain.com/"
)

docs = loader.load()


# Split documents
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)


# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


# Vector database
vectordb = FAISS.from_documents(
    documents,
    embeddings
)


# Retriever
retriever = vectordb.as_retriever()


# Convert retriever into a tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith-search",
    "Search any information about LangSmith."
)

print(retriever_tool.name)


# Final tool list
tools = [
    wiki,
    arxiv,
    retriever_tool
]

print(tools)

langsmith-search
[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Data Science 2026\\Project Repository\\AI_Agents_LangGraph\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)), ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)), StructuredTool(name='langsmith-search', description='Search any information about LangSmith.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001F6846F6C00>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001F6846F5BC0>)]


In [39]:
# ------------------------------------------------------------
# 3. LLM
# ------------------------------------------------------------

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="openai/gpt-oss-120b"
)

In [40]:
# ------------------------------------------------------------
# 4. Agent
# ------------------------------------------------------------

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)


In [41]:
# ------------------------------------------------------------
# 5. Invoke Agent
# ------------------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me about LangSmith"
            }
        ]
    }
)

print(response)


{'messages': [HumanMessage(content='Tell me about LangSmith', additional_kwargs={}, response_metadata={}, id='dd6fb9d1-6ad0-4460-9910-ad5d584e6021'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer about LangSmith. Likely a product by LangChain? LangSmith is a platform for LLM observability, tracing, evaluation. Should browse. Use langsmith-search tool.', 'tool_calls': [{'id': 'fc_ffe5c2ec-349d-4eba-976c-3c3e5adccc69', 'function': {'arguments': '{"query":"LangSmith"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 265, 'total_tokens': 335, 'completion_time': 0.148224061, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.011789366, 'prompt_tokens_details': None, 'queue_time': 0.365948358, 'total_time': 0.160013427}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', '

In [42]:
# ------------------------------------------------------------
# 6. Extract final answer
# ------------------------------------------------------------

print(
    response["messages"][-1].content
)

**LangSmith – Observability & Evaluation Platform for LLM‑Powered Applications**

---

### What is LangSmith?

LangSmith (hosted at `smith.langchain.com`) is a **full‑stack observability, evaluation, and prompt‑engineering platform** built by the LangChain team for anyone who runs production‑grade language‑model (LLM) applications—agents, chatbots, retrieval‑augmented generation pipelines, etc.  

It gives you the same kind of visibility that traditional APM tools (like Datadog or New Relic) provide for web services, but tuned to the unique lifecycle of LLM calls.

---

### Core Capabilities

| Capability | What It Does | Why It Matters |
|------------|--------------|----------------|
| **Trace Collection** | Every LLM request, tool call, and intermediate step is recorded as a *trace*. | Lets you replay exactly what happened in production, spot failures, and understand model behavior. |
| **Debugging UI** | Interactive view of a trace with the full prompt, model response, token usage, 